# Operational Supervisor — Evaluation

Standalone evaluation task for the deployed Operational Supervisor and all 6
Knowledge Assistants. Runs after `Readiness_Check` in the `all` target pipeline
so endpoints are up before we hit them.

This notebook demonstrates how to evaluate the Casper's Kitchens Operational Supervisor using **MLflow GenAI evaluation**. It covers:

1. **Evaluation datasets** — curated questions with behavioral guidelines (data-agnostic, so they survive simulator updates)
2. **LLM-as-judge scorers** — built-in `Guidelines` and `Safety` judges + custom `routing_accuracy` and `cites_specific_data` scorers
3. **Running evaluation** — `mlflow.genai.evaluate()` against the live MAS endpoint
4. **Trace-derived datasets** — how to harvest production traces and turn them into a labelled eval set

The Operational Supervisor routes questions across 7 sub-agents:
```
Operational Supervisor (MAS)
  ├── Revenue Analytics Genie    → lakeflow + simulator Delta tables
  ├── Operations Intelligence Genie → food_safety + all_events tables
  ├── Inspection Reports KA      → food safety PDF reports
  ├── Legal Complaints KA        → litigation case file PDFs
  ├── Regulatory Documents KA    → permits & certifications PDFs
  ├── Audit Findings KA          → financial / operational audit PDFs
  └── Consultancy Strategy KA    → management consulting report PDFs
```

> **Demo tip:** Run cells 1–5 before the live demo. Results will be ready to show in the MLflow Experiments UI during Part 3 of the demo script.

In [ ]:
%pip install --upgrade mlflow databricks-sdk 'mlflow[databricks]'
dbutils.library.restartPython()

In [ ]:
# (intentionally empty — was a temporary guidelines-shape diagnostic; bug
# was located in the build cell below, so the diagnostic is no longer needed.
# Safe to skip this cell.)


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os
import sys
sys.path.append("../../utils")

# Bump MLflow client-side HTTP timeouts before any deploy/judge calls.
#  • MLFLOW_DEPLOYMENT_PREDICT_TIMEOUT (default 120s): single supervisor predict.
#    Multi-domain board-deck questions can take 3–5 minutes when the supervisor
#    fans out to several sub-agents.
#  • MLFLOW_HTTP_REQUEST_TIMEOUT (default 120s): used by the LLM-judge calls
#    inside Safety / ExpectationsGuidelines / Guidelines scorers.
#  • MLFLOW_DEPLOYMENT_PREDICT_TOTAL_TIMEOUT (default 600s): total retry budget;
#    must be ≥ predict timeout.
os.environ.setdefault("MLFLOW_DEPLOYMENT_PREDICT_TIMEOUT", "600")
os.environ.setdefault("MLFLOW_DEPLOYMENT_PREDICT_TOTAL_TIMEOUT", "900")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "600")

from databricks.sdk import WorkspaceClient

try:
    CATALOG = dbutils.widgets.get("CATALOG")
except Exception:
    CATALOG = "caspersdev"

_w = WorkspaceClient()
_supervisor_tile_name = f"{CATALOG}-operational-supervisor"
_current_user = spark.sql("SELECT current_user()").collect()[0][0]


def _resolve_supervisor_endpoint(tile_name: str) -> str:
    """Return the serving endpoint name for the named supervisor.

    Uses the v2.1 SupervisorAgents list API. The legacy v2.0 path
    /api/2.0/multi-agent-supervisors only supports per-resource ops
    (POST/GET-by-id/PATCH/DELETE) — listing on it returns 'No API found'.
    """
    # Primary: v2.1 supervisor-agents list API. Items are flat dicts —
    # `display_name` is the user-facing name; `name` is the resource path.
    try:
        params = {}
        while True:
            resp = _w.api_client.do("GET", "/api/2.1/supervisor-agents", query=params)
            for sa in resp.get("supervisor_agents", []):
                if sa.get("display_name") == tile_name or sa.get("name") == tile_name:
                    ep = sa.get("endpoint_name", "")
                    if ep:
                        return ep
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"⚠️  /api/2.1/supervisor-agents lookup failed: {e}")

    # Fallback: generic tiles API
    try:
        params = {}
        while True:
            resp = _w.api_client.do("GET", "/api/2.0/tiles", query=params)
            for tile in resp.get("tiles", []):
                if tile.get("name") == tile_name:
                    ep = tile.get("serving_endpoint_name", "")
                    if ep:
                        return ep
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"⚠️  Tiles API fallback failed: {e}")

    return ""


SUPERVISOR_ENDPOINT = _resolve_supervisor_endpoint(_supervisor_tile_name)
if not SUPERVISOR_ENDPOINT:
    raise ValueError(
        f"Could not resolve supervisor endpoint for tile '{_supervisor_tile_name}'. "
        "Make sure the Operational Supervisor stage has been deployed for this catalog."
    )

print(f"Catalog:             {CATALOG}")
print(f"Supervisor endpoint: {SUPERVISOR_ENDPOINT}")


In [ ]:
import mlflow
import mlflow.genai
import pandas as pd
import json

from mlflow.deployments import get_deploy_client as _gdc
_deploy_client = _gdc("databricks")

# Each Agent Bricks Multi-Agent Supervisor gets a managed MLflow experiment under
# /Users/{user}/{mas_id}-dev-experiment.  This is THE default experiment for the
# supervisor — eval runs and production traces both land here.  No /Serving/
# companion exists for managed agent endpoints; the path below is the only place
# to look for supervisor activity.
_mas_id = SUPERVISOR_ENDPOINT.replace("-endpoint", "")
EVAL_EXPERIMENT_NAME = f"/Users/{_current_user}/{_mas_id}-dev-experiment"

mlflow.set_experiment(EVAL_EXPERIMENT_NAME)
print(f"Eval experiment: {EVAL_EXPERIMENT_NAME}")

w = _w
print(f"MLflow version: {mlflow.__version__}")


## 1. Build the Supervisor Evaluation Dataset (UC-managed)

Curated questions covering all supervisor sub-agents, with per-row `expectations.guidelines` for the `ExpectationsGuidelines` scorer.  Stored as a UC-managed `mlflow.genai.datasets` table at `{CATALOG}.evaluations.operational_supervisor_eval_dataset` so it has lineage, governance, and is reproducible across deploys.

Built inline in this notebook — no separate setup task.


In [ ]:
import json as _json
import mlflow.genai.datasets

EVAL_RECORDS = [
    {
        "inputs": {"question": "Which location has the highest order cancellation rate right now?", "expected_agent": "revenue"},
        "expectations": {
            "expected_agent": "revenue",
            "guidelines": [
                "Must name exactly one specific location as having the highest rate — not a list.",
                "Must include a numeric cancellation rate (percentage).",
                "Must not say it cannot access the data or ask for clarification.",
            ],
        },
    },
    {
        "inputs": {"question": "How does revenue compare across our locations this week?", "expected_agent": "revenue"},
        "expectations": {
            "expected_agent": "revenue",
            "guidelines": [
                "Must provide revenue figures or a ranking for multiple locations.",
                "Must reference the current week as the time period.",
                "Must not return a generic description without actual numbers.",
            ],
        },
    },
    {
        "inputs": {"question": "Which brand is generating the most revenue right now?", "expected_agent": "revenue"},
        "expectations": {
            "expected_agent": "revenue",
            "guidelines": [
                "Must name a specific brand (not a location).",
                "Must include a revenue figure or ranking.",
                "Must distinguish between brand-level and location-level performance.",
            ],
        },
    },
    {
        "inputs": {"question": "Which location needs the most operational attention right now?", "expected_agent": "operations"},
        "expectations": {
            "expected_agent": "operations",
            "guidelines": [
                "Must name one specific location with the highest risk.",
                "Must justify with at least two operational metrics (e.g. complaint rate, cancel rate, food safety).",
                "Must not give a generic answer — must reference actual data from the system.",
            ],
        },
    },
    {
        "inputs": {"question": "What happened during the Chicago food safety inspection? Were there critical violations?", "expected_agent": "inspection"},
        "expectations": {
            "expected_agent": "inspection",
            "guidelines": [
                "Must reference a specific inspection report by date or ID.",
                "Must state the inspection score and/or grade.",
                "Must explicitly state whether critical violations were found.",
                "Must cite at least one specific violation if they exist.",
            ],
        },
    },
    {
        "inputs": {"question": "Do we have any active high-risk legal cases? What is the total financial exposure?", "expected_agent": "legal"},
        "expectations": {
            "expected_agent": "legal",
            "guidelines": [
                "Must confirm whether active cases exist — cannot hedge or say data is unavailable.",
                "Must cite at least one specific case number (format: CK-XX-XXXX).",
                "Must include a risk classification (HIGH / MEDIUM / LOW) for at least one case.",
                "Must state a financial exposure amount.",
                "Must include a reminder to involve legal counsel.",
            ],
        },
    },
    {
        "inputs": {"question": "Which location has the most active legal cases?", "expected_agent": "legal"},
        "expectations": {
            "expected_agent": "legal",
            "guidelines": [
                "Must name a specific location — not 'I don\'t know' or a list of all locations.",
                "Must provide a count of active cases for at least the top location.",
                "Must distinguish active cases from settled or dismissed ones.",
            ],
        },
    },
    {
        "inputs": {"question": "Are there any permits or regulatory certificates expiring in the next 60 days?", "expected_agent": "regulatory"},
        "expectations": {
            "expected_agent": "regulatory",
            "guidelines": [
                "Must list specific document IDs or permit names — not generic descriptions.",
                "Must include expiry dates for each flagged document.",
                "Must specify which location each document applies to.",
                "If nothing is expiring, must say so explicitly with supporting evidence.",
            ],
        },
    },
    {
        "inputs": {"question": "What do our consultants recommend as the top AI investments for the next 90 days?", "expected_agent": "consultancy"},
        "expectations": {
            "expected_agent": "consultancy",
            "guidelines": [
                "Must reference a specific consulting report or firm by name.",
                "Must include at least one concrete recommendation (not just \'invest in AI\').",
                "Must include a financial metric — ROI estimate, cost, or projected saving.",
                "Must frame the answer in the 90-day horizon.",
            ],
        },
    },
    {
        "inputs": {"question": "What were the most significant audit findings this quarter?", "expected_agent": "audits"},
        "expectations": {
            "expected_agent": "audits",
            "guidelines": [
                "Must cite the auditing firm and the audit period covered.",
                "Must classify findings by severity (Critical / Significant / Minor / Informational).",
                "Must state remediation status for at least one finding.",
                "Must not conflate audit findings with food safety inspection findings.",
            ],
        },
    },
    {
        "inputs": {"question": "Give me a board deck summary: revenue performance, top operational risk, legal exposure, and one strategic recommendation.", "expected_agent": "multi"},
        "expectations": {
            "expected_agent": "multi",
            "guidelines": [
                "Must explicitly address all four domains: revenue, operations, legal, and strategy.",
                "Must include at least one concrete number or data point per domain.",
                "Must not refuse or hedge on any of the four domains.",
                "Response must be structured — not a single unbroken paragraph.",
            ],
        },
    },
]

UC_DATASET_TABLE = f"{CATALOG}.evaluations.operational_supervisor_eval_dataset"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.evaluations")

# Drop + recreate to avoid Arrow schema mismatches if a prior run wrote a
# different shape.  mlflow.genai.datasets is the source of truth from now on.
spark.sql(f"DROP TABLE IF EXISTS {UC_DATASET_TABLE}")
_ds = mlflow.genai.datasets.create_dataset(uc_table_name=UC_DATASET_TABLE)


# Pass EVAL_RECORDS to mlflow.genai.datasets natively.  An earlier revision of
# this notebook JSON-stringified every list-of-primitives field (including
# `expectations.guidelines`) on the assumption that Arrow + mlflow.genai.datasets
# couldn't round-trip ARRAY<STRING>.  That workaround actively caused the bug
# it claimed to avoid: mlflow.genai.datasets's schema treated
# `expectations.guidelines` as ARRAY<STRING>, received the JSON string we'd
# manufactured, and cast str → array by iterating characters.  Net effect:
# `guidelines` arrived in MLflow as one element per character (e.g.
# `["[", "\"", "M", "u", "s", "t", …]`) and the ExpectationsGuidelines judge
# scored against garbage.  The cell below now passes native lists; if
# mlflow.genai.datasets really can't store them, we tolerate the write failure
# and fall back to the in-memory EVAL_RECORDS (cell that loads EVAL_DATASET
# does the same).
_UC_WRITE_OK = False
try:
    _ds.merge_records(EVAL_RECORDS)
    _UC_WRITE_OK = True
    print(f"✅ Built {UC_DATASET_TABLE} ({len(EVAL_RECORDS)} rows, native ARRAY<STRING>)")
except Exception as _e:
    print(
        f"⚠️  Could not write {UC_DATASET_TABLE} natively: "
        f"{type(_e).__name__}: {_e}\n"
        "    The UC table will be empty / partially populated, but eval still\n"
        "    proceeds against the in-memory EVAL_RECORDS list in the next cells."
    )


## 2. Define the `predict_fn`

The predict function wraps the Operational Supervisor endpoint. MLflow calls it for each row in the evaluation dataset, unpacking the `inputs` dict as keyword arguments.

The supervisor is a Multi-Agent Supervisor served at a Databricks Inference Endpoint — we query it exactly like any other chat model.

In [ ]:
from mlflow.deployments import get_deploy_client as _get_deploy_client

# mlflow.deployments client is the correct way to call endpoints inside
# mlflow.genai.evaluate() — it sends plain dicts without the SDK's
# object-serialisation layer that causes "'dict' has no attribute 'as_dict'".
_deploy_client = _get_deploy_client("databricks")


def _extract_text(response: dict) -> str:
    """
    Extract the assistant text from any known MAS response format:
    - Chat Completions: {"choices": [{"message": {"content": "..."}}]}
    - Responses API:    {"output": [{"role": "assistant", "content": "..." | [...]}]}
    - Agent final:      {"final_response": "..."}
    - Plain string:     "..."
    """
    # Deploy client sometimes returns a JSON string instead of a parsed dict —
    # try to parse it before treating it as plain text.
    if isinstance(response, str):
        try:
            response = json.loads(response)
        except (json.JSONDecodeError, ValueError):
            return response  # genuinely plain text

    if not isinstance(response, dict):
        return str(response)

    # {"final_response": "..."}
    if "final_response" in response:
        return str(response["final_response"])

    # Deploy client may wrap in {"predictions": [...] or {...}}
    preds = response.get("predictions")
    if preds is not None:
        if isinstance(preds, list) and preds:
            return _extract_text(preds[0])
        if isinstance(preds, (dict, str)):
            return _extract_text(preds)

    # Chat Completions format
    if "choices" in response:
        return response["choices"][0]["message"]["content"]

    # Responses API format
    output = response.get("output", [])
    if isinstance(output, list):
        for item in output:
            if not isinstance(item, dict):
                continue
            if item.get("role") == "assistant" or item.get("type") == "message":
                content = item.get("content", "")
                if isinstance(content, str):
                    return content
                if isinstance(content, list):
                    return " ".join(
                        c.get("text", c.get("value", ""))
                        for c in content
                        if isinstance(c, dict)
                    )
    if isinstance(output, str):
        return output

    print(f"\u26a0\ufe0f  Unrecognised response format \u2014 keys: {list(response.keys())}. Returning raw string.")
    return str(response)


def predict_fn(question: str, expected_agent: str = None) -> str:
    """
    Call the Operational Supervisor MAS and return its text response.

    mlflow.genai.evaluate() calls this for each dataset row, unpacking the
    `inputs` dict as kwargs.  MAS endpoints use the agent-style `input` field
    and return either Chat Completions or Responses API format depending on
    the SDK version — _extract_text handles both.

    Per-row exceptions are caught and returned as an error string so a single
    failed call does not abort the entire evaluation run.
    """
    try:
        response = _deploy_client.predict(
            endpoint=SUPERVISOR_ENDPOINT,
            inputs={"input": [{"role": "user", "content": question}]},
        )
        return _extract_text(response)
    except Exception as e:
        print(f"⚠️  predict_fn error for question {question[:60]!r}: {e}")
        return f"[ERROR: {e}]"


print(f"✅ predict_fn defined — will call {SUPERVISOR_ENDPOINT}")

## 3. Load the Evaluation Dataset from Unity Catalog

The curated supervisor eval dataset lives in Unity Catalog at `{CATALOG}.evaluations.operational_supervisor_eval_dataset`, built by the previous cell. Reading it back from UC keeps the rest of the notebook independent of the in-memory `EVAL_RECORDS` variable, so eval runs always use the governed source of truth.

A good evaluation dataset for a multi-agent supervisor tests **agent behavior**, not specific data values.

Data changes — the simulator updates, new locations get added, legal cases resolve. Hard-coding "Chicago has the highest cancellation rate at 11–12%" would make the eval brittle and wrong the moment the data shifts.

Instead, each record defines **behavioral guidelines**: criteria that should hold true regardless of what the current data says:
- Did the agent route to the right sub-agent domain?
- Did it return a structured, data-backed response (numbers, dates, IDs)?
- Did it avoid vague hedging ("I cannot access...")?
- Did it follow the expected response format for that question type?

This also means we drop the `Correctness` scorer — that scorer compares outputs against a hardcoded `expected_response`, which is only appropriate when you have ground truth that doesn't change (like a fixed Q&A FAQ, not a live analytics system).

In [ ]:
import copy as _copy

# Use EVAL_RECORDS (in-memory, from the build cell) as the canonical eval
# data source, not a re-read of the UC table.  Reading back through
# spark.table -> toPandas -> json.loads was previously paired with the
# JSON-stringify workaround that mangled `expectations.guidelines` into a
# per-character ARRAY<STRING>; that's now removed.  We still want
# EVAL_DATASET to be a separate object the rest of the notebook can mutate
# (top-level `guidelines` lift below) without touching EVAL_RECORDS.
UC_DATASET_TABLE = f"{CATALOG}.evaluations.operational_supervisor_eval_dataset"
EVAL_DATASET = [_copy.deepcopy(r) for r in EVAL_RECORDS]

# Shape shim: EVAL_RECORDS stores guidelines under `expectations.guidelines`,
# but MLflow 3.x's ExpectationsGuidelines scorer (and cell 11's _eval_df
# construction) reads `guidelines` at the top level of each row.  Mirror it
# up so the rest of the notebook works unchanged.
for _rec in EVAL_DATASET:
    if "guidelines" not in _rec and isinstance(_rec.get("expectations"), dict):
        _gl = _rec["expectations"].get("guidelines")
        if _gl is not None:
            _rec["guidelines"] = _gl

if not EVAL_DATASET:
    raise RuntimeError(
        "EVAL_RECORDS is empty. Run the 'Build the Supervisor Evaluation "
        "Dataset' cell above before this one."
    )

print(f"Loaded {len(EVAL_DATASET)} eval rows from in-memory EVAL_RECORDS")

# Governance sanity check: confirm the UC table written by the build cell
# above also exists and has the expected row count (it's what mlflow.log_input
# below points at for lineage).  If the native write failed, this just warns —
# eval still runs against the in-memory EVAL_DATASET.
try:
    _uc_rowcount = spark.table(UC_DATASET_TABLE).count()
    if _uc_rowcount == len(EVAL_DATASET):
        print(f"   UC governance copy:  {UC_DATASET_TABLE} ({_uc_rowcount} rows) ✓")
    else:
        print(
            f"⚠️  UC governance copy:  {UC_DATASET_TABLE} has {_uc_rowcount} rows, "
            f"in-memory has {len(EVAL_DATASET)}.  Eval will use the in-memory copy."
        )
except Exception as _e:
    print(
        f"⚠️  UC governance copy:  {UC_DATASET_TABLE} not readable "
        f"({type(_e).__name__}: {_e}).  Eval will use the in-memory copy."
    )

df = pd.DataFrame(EVAL_DATASET)
print(f"Evaluation dataset: {len(df)} questions across "
      f"{df['expectations'].apply(lambda x: x['expected_agent']).nunique()} agent domains")
df[["inputs", "guidelines"]]


## 4. Define Scorers

| Scorer | Type | What it checks |
|---|---|---|
| `ExpectationsGuidelines()` | Built-in LLM judge | Per-row pass/fail criteria from `expectations.guidelines` |
| `RelevanceToQuery()` | Built-in LLM judge | Is the response directly relevant and responsive to the question asked? |
| `Safety()` | Built-in LLM judge | Does the response contain harmful or inappropriate content? |
| `routing_accuracy` | Custom `@scorer` | Does the response vocabulary match the expected sub-agent domain? |
| `cites_specific_data` | Custom `@scorer` | Does the response cite numbers/dates/IDs rather than hedge? |

> **`ExpectationsGuidelines` vs `Guidelines`:** `Guidelines()` applies the *same* criteria to every row — useful for global rules like "always respond in English". `ExpectationsGuidelines()` reads `guidelines` from each row's `expectations` dict, so each question gets its own pass/fail criteria. Since every question here has different requirements, `ExpectationsGuidelines` is correct.
>
> **`RelevanceToQuery`:** LLM judge that evaluates whether the response actually answers the question asked — catches cases where the supervisor returns a response that passes safety and guideline checks but doesn't address what the operator asked.
>
> **No `Correctness` scorer.** Correctness compares against a hardcoded `expected_response`. For a live analytics system the data changes constantly — hardcoded gold answers become false failures after every simulator update. `ExpectationsGuidelines` + `cites_specific_data` cover the same intent without brittleness.


In [ ]:
import re
try:
    from mlflow.genai.scorers import scorer, ExpectationsGuidelines, RelevanceToQuery, Safety, Guidelines
    _has_relevance = True
except ImportError:
    from mlflow.genai.scorers import scorer, ExpectationsGuidelines, Safety, Guidelines
    RelevanceToQuery = None
    _has_relevance = False
    print("⚠️  RelevanceToQuery not available in this MLflow version — skipping")
from mlflow.entities import Feedback

# ── Domain vocabulary signals ─────────────────────────────────────────────────
# Each sub-agent produces responses with characteristic vocabulary.
# We use this to infer routing without access to internal MAS trace data.
_AGENT_SIGNALS = {
    "revenue": ["revenue", "cancellation rate", "order count", "orders placed", "sales", "avg order", "weekly", "total orders"],
    "operations": ["complaint rate", "kitchen", "operational", "throughput", "busiest", "food safety grade", "cancel rate"],
    "inspection": ["inspection report", "violation", "inspector", "corrective action", "grade", "score", "health permit"],
    "legal": ["case no", "ck-", "risk level", "amount at stake", "counsel", "litigation", "plaintiff", "high risk"],
    "regulatory": ["permit", "certificate", "expiry", "regulatory", "fda", "zoning", "issuing authority"],
    "audits": ["audit", "auditor", "finding", "pwc", "deloitte", "kpmg", "significant", "critical finding"],
    "consultancy": ["consultant", "roi", "recommend", "strategy", "mckinsey", "phase 1", "investment"],
    "multi": ["revenue", "legal", "audit", "operational"],  # board summary — needs all domains
}


@scorer
def routing_accuracy(inputs: dict, outputs, expectations: dict = None) -> Feedback:
    """
    Checks whether the MAS response contains vocabulary signals consistent
    with the expected sub-agent domain for this question.
    Returns a score from 0.0 (wrong domain) to 1.0 (clearly routed correctly).
    """
    def _as_text(o):
        # Inlined so the registered/serialized scorer can resolve it at runtime.
        # Eval time hands us the string from predict_fn; prod monitoring hands us
        # the raw trace outputs (dict-shaped for the MAS endpoint).
        if o is None: return ""
        if isinstance(o, str): return o
        if isinstance(o, dict):
            v = o.get("final_response")
            if isinstance(v, str): return v
            msgs = o.get("messages")
            if isinstance(msgs, list):
                for m in reversed(msgs):
                    if isinstance(m, dict) and m.get("role") == "assistant":
                        c = m.get("content")
                        if isinstance(c, str): return c
            choices = o.get("choices")
            if isinstance(choices, list) and choices:
                try:
                    c = choices[0]["message"]["content"]
                    if isinstance(c, str): return c
                except (KeyError, TypeError):
                    pass
            output = o.get("output")
            if isinstance(output, str): return output
            if isinstance(output, list):
                parts = []
                for it in output:
                    if isinstance(it, dict):
                        c = it.get("content")
                        if isinstance(c, str): parts.append(c)
                        elif isinstance(c, list):
                            for piece in c:
                                if isinstance(piece, dict):
                                    parts.append(piece.get("text", piece.get("value", "")) or "")
                if parts: return " ".join(parts)
        return str(o)
    expected_agent = (inputs or {}).get("expected_agent") or (expectations or {}).get("expected_agent", "")
    response_lower = _as_text(outputs).lower()

    if not expected_agent or expected_agent not in _AGENT_SIGNALS:
        return Feedback(value=None, rationale="No expected_agent specified — skipped")

    signals = _AGENT_SIGNALS[expected_agent]

    if expected_agent == "multi":
        domains_present = sum(
            1 for domain_signals in _AGENT_SIGNALS.values()
            if any(s in response_lower for s in domain_signals)
        )
        score = min(1.0, domains_present / 4.0)
        rationale = f"{domains_present} agent domains detected in board summary response"
    else:
        matched = [s for s in signals if s in response_lower]
        score = min(1.0, len(matched) / max(2, len(signals) // 2))
        rationale = f"Matched signals: {matched}" if matched else "No expected-agent signals detected in response"

    return Feedback(value=score, rationale=rationale)


@scorer
def cites_specific_data(inputs: dict, outputs, expectations: dict = None) -> Feedback:
    """
    Checks whether the response cites concrete data points rather than
    giving vague or hedged answers. Operational responses must have numbers.
    """
    import re  # imported inside the function so the registered/serialized scorer can resolve it at runtime
    def _as_text(o):
        # Inlined so the registered/serialized scorer can resolve it at runtime.
        # Eval time hands us the string from predict_fn; prod monitoring hands us
        # the raw trace outputs (dict-shaped for the MAS endpoint).
        if o is None: return ""
        if isinstance(o, str): return o
        if isinstance(o, dict):
            v = o.get("final_response")
            if isinstance(v, str): return v
            msgs = o.get("messages")
            if isinstance(msgs, list):
                for m in reversed(msgs):
                    if isinstance(m, dict) and m.get("role") == "assistant":
                        c = m.get("content")
                        if isinstance(c, str): return c
            choices = o.get("choices")
            if isinstance(choices, list) and choices:
                try:
                    c = choices[0]["message"]["content"]
                    if isinstance(c, str): return c
                except (KeyError, TypeError):
                    pass
            output = o.get("output")
            if isinstance(output, str): return output
            if isinstance(output, list):
                parts = []
                for it in output:
                    if isinstance(it, dict):
                        c = it.get("content")
                        if isinstance(c, str): parts.append(c)
                        elif isinstance(c, list):
                            for piece in c:
                                if isinstance(piece, dict):
                                    parts.append(piece.get("text", piece.get("value", "")) or "")
                if parts: return " ".join(parts)
        return str(o)
    response = _as_text(outputs)

    patterns = [
        r'\d+\.?\d*\s*%',                        # percentages
        r'\$\s*[\d,]+',                           # dollar amounts
        r'CK-\d+-\d+',                            # legal case numbers
        r'\d{4}-\d{2}-\d{2}',                     # ISO dates
        r'\b(?:score|grade)\s*(?:of\s*)?\d+',     # inspection scores
        r'\b\d+x\b',                              # ROI multiples (e.g. 3.2x)
        r'\b\d+\s+(?:cases?|orders?|locations?|findings?)',  # count phrases
    ]

    found = [pat for pat in patterns if re.search(pat, response, re.IGNORECASE)]

    hedges = ["i don't have access", "i cannot provide", "i'm unable", "consult your", "please check with"]
    is_hedged = any(h in response.lower() for h in hedges)

    if is_hedged and len(found) < 2:
        return Feedback(value=0.0, rationale="Response is hedged with no concrete data — agent likely failed to retrieve")

    score = min(1.0, len(found) / 3.0)
    return Feedback(value=score, rationale=f"Data patterns found: {len(found)} types — {found}")


print("Scorers defined:")
print("  - ExpectationsGuidelines  (LLM judge: per-row pass/fail from expectations.guidelines)")
print("  - RelevanceToQuery        (LLM judge: is the response directly relevant to the question?)")
print("  - Safety                  (LLM judge: harmful/inappropriate content check)")
print("  - Guidelines              (LLM judge: global guideline for trace-derived eval — trace eval only)")
print("  - routing_accuracy        (custom: did MAS route to the right domain?)")
print("  - cites_specific_data     (custom: does the answer contain concrete numbers?)")


## 5. Run Evaluation

`mlflow.genai.evaluate()` calls `predict_fn` for each row, applies all scorers, and logs everything to the MLflow experiment as a named run.

> This will make **10 calls** to the Operational Supervisor endpoint. Each call takes ~30–120 seconds depending on agent routing. Expect total runtime of **5–15 minutes**.

In [ ]:
# Log the evaluation dataset to MLflow so it's visible in the Experiments UI.
# This makes the dataset a first-class artifact linked to the run — reviewers
# can see exactly which questions were used for evaluation without reading code.
# Log the UC table itself as the run's input dataset, so the run carries
# UC lineage to {CATALOG}.evaluations.operational_supervisor_eval_dataset
# (matches refund/complaint pattern). UC_DATASET_TABLE is defined in the
# "Build the Evaluation Dataset" cell above.
_mlflow_dataset = mlflow.data.from_spark(
    spark.table(UC_DATASET_TABLE),
    table_name=UC_DATASET_TABLE,
)

# _mas_id is defined in cell 3 (e.g. "mas-e693df14")
_eval_run_name = f"{_mas_id}-dev-experiment"

# Resolve the production supervisor_instructions prompt version so eval runs
# can be tagged with it.  Lets a failing scorer point at exactly which prompt
# revision regressed.  CATALOG is defined in cell 2.
import mlflow
import mlflow.genai

# Required: without databricks-uc the registry client hits workspace MLflow,
# where 3-part names are opaque strings and load_prompt raises NotFound.
mlflow.set_registry_uri("databricks-uc")

SUPERVISOR_PROMPT_URI = f"prompts:/{CATALOG}.prompts.supervisor_instructions@production"
try:
    SUPERVISOR_PROMPT_VERSION = str(mlflow.genai.load_prompt(SUPERVISOR_PROMPT_URI).version)
    print(f"  Resolved {SUPERVISOR_PROMPT_URI} -> v{SUPERVISOR_PROMPT_VERSION}")
except Exception as _exc:
    SUPERVISOR_PROMPT_VERSION = "unknown"
    print(f"  Could not resolve {SUPERVISOR_PROMPT_URI}: {type(_exc).__name__}: {_exc}")

with mlflow.start_run(run_name=_eval_run_name):
    mlflow.log_input(_mlflow_dataset, context="eval")
    mlflow.log_param("prompt_uri", SUPERVISOR_PROMPT_URI)
    mlflow.log_param("prompt_version", SUPERVISOR_PROMPT_VERSION)
    results = mlflow.genai.evaluate(
        predict_fn=predict_fn,
        data=EVAL_DATASET,
        scorers=[
            ExpectationsGuidelines(),
            *([RelevanceToQuery()] if _has_relevance else []),
            Safety(),
            routing_accuracy,
            cites_specific_data,
        ],
    )

print("\n✅ Evaluation complete")
# Diagnostics: inspect the results object so we know its shape
print(f"results type: {type(results)}")
print(f"results.metrics: {getattr(results, 'metrics', 'N/A')}")
_tables = getattr(results, 'tables', None)
if _tables is not None:
    if isinstance(_tables, dict):
        print(f"results.tables keys: {list(_tables.keys())}")
        for _k, _v in _tables.items():
            try:
                print(f"  [{_k!r}] shape={_v.shape}, cols={_v.columns.tolist()}")
            except Exception:
                print(f"  [{_k!r}] type={type(_v)}")
    else:
        print(f"results.tables type: {type(_tables)}, value: {_tables}")
else:
    print("results has no .tables attribute")
    print(f"results attrs: {[a for a in dir(results) if not a.startswith('_')]}")


# ── Surface per-row scorer errors ─────────────────────────────────────────────
# When mlflow.genai.evaluate prints "Failure summary: 'X': N/N failed", the
# actual error messages live in the assessments. Print them inline so we don't
# need to navigate the MLflow UI to root-cause scorer failures.
def _surface_scorer_errors(results):
    try:
        tables = getattr(results, "tables", None) or {}
        eval_df = tables.get("eval_results") if isinstance(tables, dict) else None
        if eval_df is None or eval_df.empty:
            return
        err_cols = [c for c in eval_df.columns if c.endswith("/error")]
        if err_cols:
            for c in err_cols:
                errs = eval_df[c].dropna()
                errs = errs[errs.astype(str).str.strip().ne("")]
                if errs.empty:
                    continue
                scorer = c[: -len("/error")]
                print(f"\n\u26a0\ufe0f  {scorer}: {len(errs)}/{len(eval_df)} rows errored. First 3:")
                for i, e in errs.head(3).items():
                    print(f"  row {i}: {str(e)[:320]}")
            return
        # Fallback: walk an `assessments` / `feedback` list column
        for col in ("assessments", "feedback"):
            if col not in eval_df.columns:
                continue
            from collections import Counter
            counts, samples = Counter(), {}
            for assess_list in eval_df[col]:
                if not isinstance(assess_list, (list, tuple)):
                    continue
                for a in assess_list:
                    err = getattr(a, "error", None)
                    if err is None and isinstance(a, dict):
                        err = a.get("error")
                    if err:
                        name = getattr(a, "name", None)
                        if name is None and isinstance(a, dict):
                            name = a.get("name", "?")
                        counts[name] += 1
                        samples.setdefault(name, str(err)[:320])
            for name, n in counts.most_common():
                print(f"\n\u26a0\ufe0f  {name}: {n} row(s) errored. Example: {samples[name]}")
            break
    except Exception as _se:
        print(f"(could not surface scorer errors: {_se})")


_surface_scorer_errors(results)


## 6. Inspect Results

The results table shows one row per question × scorer. Use it to identify which questions the supervisor handled poorly and which agents need tuning.

In [ ]:
# Results as a pandas DataFrame
# Safely access the eval results — MLflow API shape varies by version
import pandas as _pd

results_df = _pd.DataFrame()
try:
    if hasattr(results, "tables") and isinstance(results.tables, dict):
        results_df = results.tables.get("eval_results", _pd.DataFrame())
        if results_df.empty and results.tables:
            results_df = next(iter(results.tables.values()), _pd.DataFrame())
    elif hasattr(results, "_results_df"):
        results_df = results._results_df
    elif hasattr(results, "to_pandas"):
        results_df = results.to_pandas()
except Exception as _e:
    print(f"⚠️  Could not access eval results table: {_e}")

print(f"Results shape: {results_df.shape}")
print(f"Available columns: {results_df.columns.tolist()}")

if results_df.empty:
    print("⚠️  Empty results DataFrame — evaluation may not have completed")
else:
    # Detect question column
    _q_col = next(
        (c for c in results_df.columns
         if c in ("inputs/question", "request", "question")
         or (c == "inputs" and not c.endswith("/"))),
        results_df.columns[0],
    )
    # Detect outputs column
    _out_col = next(
        (c for c in results_df.columns
         if c in ("outputs", "response", "output")),
        None,
    )
    # Scorer columns — cover both MLflow 2.x and 3.x naming conventions
    _scorer_prefixes = ["guidelines", "expectations_guidelines", "correctness",
                        "safety", "routing", "cites"]
    scorer_cols = [c for c in results_df.columns
                   if any(c.startswith(s) for s in _scorer_prefixes)]
    print(f"Question col: {_q_col!r} | Outputs col: {_out_col!r} | Scorer cols: {scorer_cols}")

    _display_cols = [_q_col] + ([_out_col] if _out_col else []) + scorer_cols
    try:
        display(
            results_df[_display_cols]
            .rename(columns={_q_col: "question"})
            .assign(question=lambda df: df["question"].astype(str).str[:80] + "...")
        )
    except Exception as _e:
        print(f"⚠️  Display failed ({_e}); showing full DataFrame:")
        display(results_df)


In [ ]:
# ── Score summary ─────────────────────────────────────────────────────────────
if not scorer_cols:
    print("⚠️  No scorer columns found — check column names above")
else:
    numeric_scorer_cols = [c for c in scorer_cols if "score" in c or "value" in c.lower()]
    if not numeric_scorer_cols:
        numeric_scorer_cols = scorer_cols

    try:
        summary = results_df[numeric_scorer_cols].apply(
            lambda col: _pd.to_numeric(col, errors="coerce")
        ).agg(["mean", "min", "max"]).T
        summary.columns = ["mean", "min", "max"]
        summary["mean"] = summary["mean"].round(3)
        print("Scorer summary (higher = better):")
        display(summary.sort_values("mean"))
    except Exception as _e:
        print(f"⚠️  Summary failed ({_e})")
        display(results_df[numeric_scorer_cols])


In [ ]:
# ── Flag failing questions ────────────────────────────────────────────────────
# Questions where the cites_specific_data score is 0 indicate the agent
# probably failed to retrieve data (hedged response with no numbers).

cites_col = next((c for c in scorer_cols if "cites" in c), None)
if cites_col:
    failing = results_df[results_df[cites_col] < 0.3][[_q_col, cites_col]]
    if len(failing):
        print(f"⚠️  {len(failing)} question(s) returned hedged/empty responses:")
        for _, row in failing.iterrows():
            _q_val = row[_q_col]
            if isinstance(_q_val, dict):
                _q_val = _q_val.get("question", str(_q_val))
            print(f"  - {str(_q_val)[:100]}")
    else:
        print("✅ All questions returned data-backed responses")

## 7. Build an Evaluation Dataset from Production Traces

In addition to a curated dataset, you can harvest **real operator questions** from production traces and turn them into an evaluation dataset. This is the recommended workflow for continuous quality monitoring:

```
Production traffic → MLflow traces → filter interesting traces → add ground truth → eval dataset
```

The Operational Supervisor endpoint automatically writes traces to MLflow — no instrumentation code needed.

In [ ]:
# Production traces auto-log to the same managed experiment we set in cell 3
# (/Users/{user}/{mas_id}-dev-experiment).  The eval runs above are tagged with
# eval_run_id; production traces have no eval_run_id, so we filter on that.

_exp = mlflow.get_experiment_by_name(EVAL_EXPERIMENT_NAME)
if _exp is not None:
    experiment_id = _exp.experiment_id
    print(f"Production traces experiment: {EVAL_EXPERIMENT_NAME} ({experiment_id})")
else:
    experiment_id = None
    print(f"⚠️  Experiment {EVAL_EXPERIMENT_NAME} not found.")
    print("  Run the cells above to fire the first supervisor traces.")


In [ ]:
if experiment_id:
    # ── Fetch recent traces ───────────────────────────────────────────────────
    # Use the MLflow search_traces API to pull recent successful traces.
    # Filter: only OK status (skip errors), limit to last 50 traces.
    #
    # NOTE: mlflow.search_traces() returns a *pandas DataFrame* by default, not
    # a list of Trace objects. The DataFrame already has `request`/`response`
    # columns extracted from the root span, so we don\'t need to dig into
    # `trace.data.spans` ourselves. Iterating with iterrows() is the correct
    # pattern; iterating `for x in df:` would walk column names instead of rows.

    traces_df_raw = mlflow.search_traces(
        experiment_ids=[experiment_id],
        filter_string="status = \'OK\'",
        max_results=50,
        order_by=["timestamp DESC"],
    )

    print(f"Found {len(traces_df_raw)} traces from the Operational Supervisor experiment")

    # ── Extract inputs and outputs ────────────────────────────────────────────
    trace_records = []
    if len(traces_df_raw) > 0:
        for _, row in traces_df_raw.iterrows():
            try:
                # Columns are documented at:
                # https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.search_traces
                # `request` and `response` are JSON-encoded strings of the root span\'s
                # inputs/outputs respectively. Both Chat Completions ({"messages": [...]})
                # and Responses API ({"input": [...]}) shapes appear in practice.
                request_raw = row.get("request")
                response_raw = row.get("response")

                inputs_raw = (
                    _json.loads(request_raw)
                    if isinstance(request_raw, str) and request_raw.strip()
                    else (request_raw or {})
                )
                outputs_raw = (
                    _json.loads(response_raw)
                    if isinstance(response_raw, str) and response_raw.strip()
                    else (response_raw or {})
                )

                # Find the user\'s question across both API shapes.
                messages = inputs_raw.get("messages") or inputs_raw.get("input") or []
                question = next(
                    (m.get("content") for m in messages if m.get("role") == "user"),
                    None
                )

                # Extract assistant\'s response text via the existing helper.
                response_text = _extract_text(outputs_raw)

                if question and response_text:
                    trace_records.append({
                        "trace_id": row.get("trace_id") or row.get("request_id") or "?",
                        "timestamp": row.get("timestamp_ms"),
                        "latency_ms": row.get("execution_time_ms"),
                        "question": question,
                        "response": response_text[:500],
                    })
            except Exception as _te:
                tid = row.get("trace_id") or row.get("request_id") or "?"
                print(f"  \u26a0\ufe0f  Skipped trace {tid}: {_te}")
                continue

    traces_df = pd.DataFrame(trace_records)
    print(f"\nExtracted {len(traces_df)} valid question/response pairs")
    if not traces_df.empty:
        display(traces_df[["question", "latency_ms", "response"]].head(10))
    else:
        print("No valid traces extracted yet — send some questions through the Operational Dashboard app, then re-run this section.")
else:
    print("Skipping trace fetch — no experiment found yet.")
    traces_df = pd.DataFrame()


In [ ]:
trace_eval_data = []

if not traces_df.empty:
    # ── Identify slow traces ──────────────────────────────────────────────────
    # Questions that took > 60s may indicate routing to a cold endpoint or a
    # hard question that caused agent confusion.
    slow_threshold_ms = 60_000
    slow = traces_df[traces_df["latency_ms"] > slow_threshold_ms].copy()
    print(f"Slow traces (>{slow_threshold_ms/1000:.0f}s): {len(slow)}")
    if not slow.empty:
        display(slow[["question", "latency_ms"]].sort_values("latency_ms", ascending=False))

    # ── Build evaluation dataset from traces ──────────────────────────────────
    # Real operator questions — no per-row guidelines.
    # A global guideline is passed to Guidelines() at eval time (cell below).
    trace_eval_data = [
        {"inputs": {"question": row["question"]}}
        for _, row in traces_df.head(20).iterrows()
    ]

    print(f"\nTrace-derived evaluation dataset: {len(trace_eval_data)} records")
    print("Sample question:", trace_eval_data[0]["inputs"]["question"][:100])


In [ ]:
# Global quality bar applied to every trace-derived question.
# Guidelines() applies the same criteria to all rows — correct for
# unsupervised traffic where we have no per-question expected behaviour.
_TRACE_GUIDELINE = (
    "Response must include specific data points (numbers, locations, dates, or names). "
    "Response must not be a generic hedge or refusal. "
    "Response must be directly relevant to the question asked."
)

if not traces_df.empty and trace_eval_data:
    with mlflow.start_run(run_name=f"{_mas_id}-dev-trace-experiment"):
        mlflow.log_param("prompt_uri", SUPERVISOR_PROMPT_URI)
        mlflow.log_param("prompt_version", SUPERVISOR_PROMPT_VERSION)
        trace_results = mlflow.genai.evaluate(
            predict_fn=predict_fn,
            data=trace_eval_data,
            scorers=[
                Guidelines(guidelines=_TRACE_GUIDELINE),
                *([RelevanceToQuery()] if _has_relevance else []),
                Safety(),
                routing_accuracy,
                cites_specific_data,
            ],
        )

    print("\u2705 Trace-based evaluation complete")
    try:
        _tr_df = trace_results.tables.get("eval_results") if isinstance(
            getattr(trace_results, "tables", None), dict
        ) else None
        display(_tr_df if _tr_df is not None else trace_results)
    except Exception as _e:
        print(f"\u26a0\ufe0f  Could not display trace results ({_e})")
else:
    print("Skipping trace evaluation — no trace data available.")


## 8. Comparing Runs

Use the MLflow Experiments UI to compare runs. The two runs created by this notebook are:

| Run | Dataset | What it tells you |
|---|---|---|
| `operational-supervisor-full-eval` | 10 curated Q&A pairs with ground truth | How well the supervisor handles the demo script questions |
| `operational-supervisor-trace-eval` | Real operator questions from production traces | General quality over real traffic |

### How to interpret the scorer results

| Score | Meaning | Action |
|---|---|---|
| `guidelines` < 0.5 | Answer doesn't satisfy the stated criteria | Improve the sub-agent's `instructions` in `knowledge_agents.ipynb` or the supervisor's `instructions` in `operational_supervisor.ipynb` |
| `correctness` < 0.5 | Factually wrong or hallucinated answer | Check the document corpus in the UC Volume — documents may be missing or outdated |
| `routing_accuracy` < 0.5 | MAS routed to the wrong agent | Improve the `description` field for that sub-agent in `operational_supervisor.ipynb` |
| `cites_specific_data` = 0.0 | Hedged/empty response | Sub-agent had a retrieval failure — check endpoint status and UC Volume contents |
| `safety` < 1.0 | Potentially harmful content in response | Review the full response — this is rare but warrants investigation |

### Finding low-quality traces to fix

```python
# Find all traces where the agent failed to cite specific data
mlflow.search_traces(
    experiment_ids=[experiment_id],
    filter_string="status = 'OK'",
    max_results=100,
)
# Then tag the failing ones for your eval dataset:
# mlflow.update_trace(request_id=trace_id, tags={"quality": "low", "issue": "no-data"})
```

### Next steps

1. **MemAlign** — after running this notebook, use `mlflow.genai.align()` to calibrate the `Guidelines` judge to match your specific definition of a good operational response
2. **Production monitoring** — link the MAS experiment to a Unity Catalog table and set up a scheduled evaluation job to track quality over time
3. **Prompt optimization** — use `mlflow.genai.optimize_prompts()` with the evaluation dataset to automatically improve the supervisor's `instructions` based on failing scores

In [ ]:
# ── Quick link to MLflow Experiments UI ───────────────────────────────
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
try:
    _exp = mlflow.get_experiment_by_name(EVAL_EXPERIMENT_NAME)
    exp_id = _exp.experiment_id if _exp else None
except Exception:
    exp_id = None
if exp_id:
    print(f"View results in MLflow:")
    print(f"   {host}/ml/experiments/{exp_id}")


## 9. Register Production Scorers (continuous sampling)

The `evaluate()` runs above are one-shot, but the dashboard also needs continuous quality monitoring on every live trace.  This section registers the same scorers via `Scorer.register()` + `Scorer.start()` against the managed Agent Bricks experiments — so every production request gets graded automatically.

- **Supervisor:** `/Users/{user}/{mas_id}-dev-experiment` (same experiment as the `evaluate()` runs above).
- **KAs:** the per-tile experiment exposed as `tile.mlflow_experiment_id` from the tile API. No path construction — we read the id directly from the tile.

Covers the **Operational Supervisor** and all **6 Knowledge Assistants**.  Refund + Complaint custom agents are scored by their own stage notebooks (`stages/refund_evaluation.ipynb`, `stages/complaint_evaluation.ipynb`).


In [ ]:
# ── Shared helpers for production scorer registration ───────────────────────
from mlflow.genai.scorers import Safety, RelevanceToQuery, Guidelines, ScorerSamplingConfig
from mlflow.genai import get_scorer
import time as _time_reg

try:
    from mlflow.genai.scorers import RetrievalGroundedness
    _HAS_RG = True
except ImportError:
    RetrievalGroundedness = None
    _HAS_RG = False
    print("⚠️  RetrievalGroundedness not available — KAs will get 4 scorers instead of 5")


def _register_scorer_on(exp_id: str, scorer_obj, name: str, sample_rate: float = 1.0):
    """Register + start a scorer on the given experiment id, robust against the
    eventual-consistency window between register() and start() on the Databricks
    side that occasionally surfaces as `ValueError: No registered scorer found`."""
    sampling = ScorerSamplingConfig(sample_rate=sample_rate)

    def _server_get():
        try:
            return get_scorer(name=name, experiment_id=exp_id)
        except Exception:
            return None

    target = _server_get()
    action = "restarted" if target is not None else "registered"
    if target is None:
        target = scorer_obj.register(name=name, experiment_id=exp_id)

    last_err = None
    for attempt in range(4):
        try:
            target.start(name=name, experiment_id=exp_id, sampling_config=sampling)
            icon = "✅" if action == "registered" else "↺"
            print(f"  {icon} {name} — {action} at {sample_rate:.0%}")
            return target
        except Exception as ve:
            if "No registered scorer" not in str(ve):
                raise
            last_err = ve
            wait_s = (attempt + 1) * 2
            print(f"  ⏳  {name} not yet visible (attempt {attempt + 1}/4) — waiting {wait_s}s")
            _time_reg.sleep(wait_s)
            refetched = _server_get()
            if refetched is not None:
                target = refetched
            else:
                target = scorer_obj.register(name=name, experiment_id=exp_id)
                action = "registered"
    raise last_err if last_err else RuntimeError(f"Could not start scorer {name!r}")


In [ ]:
# ── Supervisor scorers ──────────────────────────────────────────────────────
# Same managed experiment used for evaluate() runs above (/Users/{user}/{mas_id}-dev-experiment).
print(f"Registering scorers on: {EVAL_EXPERIMENT_NAME}\n")

_SUP_EXP_ID = mlflow.get_experiment_by_name(EVAL_EXPERIMENT_NAME).experiment_id

_register_scorer_on(_SUP_EXP_ID, Safety(),           name="safety",             sample_rate=1.0)
_register_scorer_on(_SUP_EXP_ID, RelevanceToQuery(), name="relevance_to_query", sample_rate=1.0)
_register_scorer_on(
    _SUP_EXP_ID,
    Guidelines(
        guidelines=(
            "The response must include specific data points such as numbers, percentages, "
            "dates, case numbers, or named locations. "
            "The response must not be a generic hedge or refusal (e.g. 'I don\'t have access'). "
            "The response must directly answer the question asked."
        ),
    ),
    name="operational_quality",
    sample_rate=1.0,
)
_register_scorer_on(_SUP_EXP_ID, routing_accuracy,    name="routing_accuracy",    sample_rate=1.0)
_register_scorer_on(_SUP_EXP_ID, cites_specific_data, name="cites_specific_data", sample_rate=1.0)

print(f"\n✅ Supervisor scorers active on {EVAL_EXPERIMENT_NAME}")


In [ ]:
# ── Knowledge Assistant scorers (loop over all KAs from uc_state) ──────────
# Each KA tile owns a managed MLflow experiment exposed by the tile API as
# `mlflow_experiment_id`.  That's the default experiment Databricks created for
# the tile — no /Serving/ companion, no path construction.  We fetch the id
# directly from the tile and register scorers against it.
import json as _json_ka

_KA_QUALITY = (
    "The response must directly answer the question with concrete information from the document corpus. "
    "It must not be a generic refusal or 'I don\'t have access' hedge. "
    "It must reference specific documents, IDs, dates, or numbers when relevant."
)
_KA_CITES_SOURCE = (
    "The response must cite at least one specific source from the document corpus — "
    "for example a document ID, report date, case number (e.g. CK-XX-XXXX), permit ID, "
    "audit ID, location name, or brand name. A response that paraphrases without citation fails."
)


def _ka_experiment_id(tile_id: str):
    """Return the KA tile's managed MLflow experiment id, or None if missing."""
    if not tile_id:
        return None
    try:
        tile = w.api_client.do("GET", f"/api/2.0/tiles/{tile_id}")
        eid = tile.get("mlflow_experiment_id")
        return str(eid) if eid else None
    except Exception as e:
        print(f"     tile API lookup failed: {e}")
        return None


def _register_ka_scorers(exp_id: str):
    summary = []
    _register_scorer_on(exp_id, Safety(),           name="safety",             sample_rate=1.0); summary.append("safety")
    _register_scorer_on(exp_id, RelevanceToQuery(), name="relevance_to_query", sample_rate=1.0); summary.append("relevance_to_query")
    if _HAS_RG:
        _register_scorer_on(exp_id, RetrievalGroundedness(), name="retrieval_groundedness", sample_rate=1.0); summary.append("retrieval_groundedness")
    _register_scorer_on(exp_id, Guidelines(guidelines=_KA_CITES_SOURCE), name="cites_source", sample_rate=1.0); summary.append("cites_source")
    _register_scorer_on(exp_id, Guidelines(guidelines=_KA_QUALITY),      name="ka_quality",   sample_rate=1.0); summary.append("ka_quality")
    return summary


_ka_rows = []
try:
    _ka_rows = spark.sql(f"""
        SELECT resource_data
        FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'knowledge_assistants'
        ORDER BY created_at DESC
    """).collect()
except Exception as _e:
    print(f"⚠️  Could not list KAs from uc_state: {_e}")

_seen, _ka_results = set(), []
for _row in _ka_rows:
    _info = _json_ka.loads(_row.resource_data)
    _dn = _info.get("name", "")
    _tid = _info.get("tile_id", "")
    if not _tid or _tid in _seen:
        continue
    _seen.add(_tid)
    print(f"\n🧠 {_dn} (tile={_tid})")
    _exp_id = _ka_experiment_id(_tid)
    if _exp_id is None:
        print(f"   ⏭   tile {_tid} has no mlflow_experiment_id — skipping")
        _ka_results.append((_dn, _tid, None))
        continue
    print(f"   experiment_id: {_exp_id}")
    summary = _register_ka_scorers(_exp_id)
    print(f"   ✅ {len(summary)} scorers active: {', '.join(summary)}")
    _ka_results.append((_dn, _tid, summary))

print(f"\n✅ KA scorer registration complete — {sum(1 for r in _ka_results if r[2] is not None)}/{len(_ka_results)} KAs scored")
